In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

import os
for dirname, _, filenames in os.walk('/kaggle/input/pickle-ieee'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/pickle-ieee/custom.css
/kaggle/input/pickle-ieee/__results__.html
/kaggle/input/pickle-ieee/Train.pkl
/kaggle/input/pickle-ieee/Test.pkl


In [2]:
train = pd.read_pickle('/kaggle/input/pickle-ieee/Train.pkl')
test = pd.read_pickle('/kaggle/input/pickle-ieee/Test.pkl')

In [3]:
y = train['isFraud']
del train['isFraud']

# Model

**Finding best parameters using RandomizedSearchCV**

In [4]:
import lightgbm as lgbm

In [5]:
from sklearn.model_selection import RandomizedSearchCV

In [6]:
params = {
    'num_leaves': [500,600,700,800],
    'feature_fraction': list(np.arange(0.1,0.5,0.1)),
    'bagging_fraction': list(np.arange(0.1,0.5,0.1)),
    'min_data_in_leaf': [100,120,140,160],
    'learning_rate': [0.05],
    'reg_alpha': list(np.arange(0.1,0.5,0.1)),
    'reg_lambda': list(np.arange(0.1,0.5,0.1)),
}

In [7]:
model = lgbm.LGBMClassifier(random_state=42,metric='auc',verbosity=-1,objective='binary',max_depth=-1)

In [8]:
grid = RandomizedSearchCV(model,param_distributions=params,n_iter=15,cv=3,scoring='roc_auc')

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    train, y, test_size=0.15, random_state=42)

In [11]:
grid.fit(X_train,y_train)

RandomizedSearchCV(cv=3, error_score='raise-deprecating',
                   estimator=LGBMClassifier(boosting_type='gbdt',
                                            class_weight=None,
                                            colsample_bytree=1.0,
                                            importance_type='split',
                                            learning_rate=0.1, max_depth=-1,
                                            metric='auc', min_child_samples=20,
                                            min_child_weight=0.001,
                                            min_split_gain=0.0,
                                            n_estimators=100, n_jobs=-1,
                                            num_leaves=31, objective='binary',
                                            random_state=42, re...
                                                             0.30000000000000004,
                                                             0.4],
                     

In [12]:
grid.best_params_

{'reg_lambda': 0.2,
 'reg_alpha': 0.4,
 'num_leaves': 800,
 'min_data_in_leaf': 120,
 'learning_rate': 0.05,
 'feature_fraction': 0.4,
 'bagging_fraction': 0.4}

In [13]:
grid.best_score_

0.9536176789949288

In [14]:
from sklearn.metrics import classification_report, roc_auc_score

In [15]:
print(classification_report(y_test,grid.predict(X_test)))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99     85430
           1       0.95      0.54      0.69      3151

    accuracy                           0.98     88581
   macro avg       0.97      0.77      0.84     88581
weighted avg       0.98      0.98      0.98     88581



In [16]:
print(roc_auc_score(y_test,grid.predict_proba(X_test)[:,1]))

0.9642542720673093
